# Week 2, Lab 2 — Tools with `@function_tool`


In [1]:
WEEK = 'Week 2'
LAB = 'Lab 2 — function tools'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 2 / Lab 2 — function tools
Backend: ollama
Need Ollama running: `ollama serve` and `ollama pull llama3.2:1b`
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [2]:
cfg = openai_client_kwargs()
print(cfg)

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, handoff

client = AsyncOpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])
model = OpenAIChatCompletionsModel(model=cfg["model"], openai_client=client)


{'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama', 'model': 'qwen2.5:3b'}


In [9]:
from agents import function_tool, set_tracing_disabled

set_tracing_disabled(True)


@function_tool
def calculator_tool(expression: str) -> str:
    """Evaluate a basic arithmetic expression."""
    return calculator(expression)

@function_tool
def lookup_fact_tool(topic: str) -> str:
    """Look up a local fact about agentic AI."""
    return lookup_fact(topic)

@function_tool
def today_date_tool() -> str:
    """ Retrieves today's date"""
    return today_date()


agent = Agent(
    name="ToolTutor",
    instructions="Use tools for math and for course-topic facts. Be brief.",
    model=model,
    tools=[calculator_tool, lookup_fact_tool, today_date_tool],
)

r1 = await Runner.run(agent, "evaulate 45 * 12 + 30")
print("MATH:", r1.final_output)
r2 = await Runner.run(agent, "What is langchain")
print("FACT:", r2.final_output)
r3 = await Runner.run(agent,"What is today's date")
print("Today Date:",r3.final_output)

MATH: The result of evaluating the expression \(45 \times 12 + 30\) is 570.
FACT: LangChain is described as a toolkit for prompts, chains, retrievers, and tool-using agents. It's designed to help with AI tasks involving these components.
Today Date: Today's date is September 12, 2026.


## Exercise\n\nAdd `today_date` as a third tool. If the small model skips tools, insist in instructions: you MUST call a tool.\n\n**Next:** handoffs.
